# PubChem BioAssays for *Mycobacterium tuberculosis*

This notebook processes PubChem BioAssay data associated with *Mycobacterium tuberculosis* and its related taxa.

1. Loads the exported TaxID–AID table  
2. Cleans the dataset and removes taxa with no BioAssays  
3. Expands the pipe-separated AID lists into individual assay records  
4. Produces a tidy TaxID–AID table  
5. Summarizes:
   - number of unique taxonomy entries  
   - number of unique BioAssays  
   - number of assays per taxon  

This serves as the **first step** for building a complete organism-level
BioAssay retrieval pipeline for antimicrobial pathogens.

## 0. Setup

In [44]:
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
DATA_DIR = NOTEBOOK_DIR.parent / "data" / "raw"

## 1. Load the TaxID–AID Mapping

PubChem currently does not provide a stable programmatic endpoint to query
taxonomy-linked BioAssays directly from an organism name. Therefore, the initial dataset used here was downloaded manually from the
PubChem interface:

Search → “Mycobacterium tuberculosis” → Taxonomy →  Actions → BioAssays → Download: Summary (Search Results).

The downloaded file contains:
- PubChem Taxonomy IDs (TaxID)
- Scientific names
- Pipe-separated BioAssay IDs (AID) linked to each taxon

In [45]:
df_raw = pd.read_csv(DATA_DIR / "PubChem_taxonomy_text_Mycobacterium tuberculosis.csv")
df_raw.head()

,Taxonomy_ID,Data_Source,type,Taxonomy_Name,Synonyms,Linked_BioAssays,Linked_Proteins,Linked_Genes,Linked_Pathways,pmids,Identifiers,dois,pmcids,pclids,citations
0,1773,BioAssay|Pathway|Patent|Cooccurrence,1,Mycobacterium tuberculosis,Bacillus tuberculosis|'Bacillus tuberculosis' ...,375|838|1811|38730|38912|38913|40463|40464|103...,1G2O_A|1G2O_B|1I80_B|1MRN_A|1MRS_A|1N3I_A|1N3I...,NaN,Reactome:R-MTU-868688|Reactome:R-MTU-870331|Re...,29205127,ITIS:963806|MeSH:D009169|NCIt:C76370|UniProt:1...,10.1099/ijsem.0.002507,NaN,27028446,"Riojas MA, McGough KJ, Rider-Riojas CJ, Rastog..."
1,1765,BioAssay|Pathway|Cooccurrence,1,Mycobacterium tuberculosis variant bovis,Mycobacterium bovis Karlson and Lessel 1970|My...,143878|143879|143880|143993|143994|143995|1439...,P46829|Q93SP7,NaN,NaN,10618079|11931153|14657105|29205127,MeSH:D009163|NCIt:C85545|Wikidata:Q133447,10.1099/00207713-52-2-433|10.1099/ijs.0.02532-...,PMC86043,4311809|11807723|26781904|27028446,"Riojas MA, McGough KJ, Rider-Riojas CJ, Rastog..."
2,652616,BioAssay|Pathway|Patent|Cooccurrence,1,Mycobacterium tuberculosis str. Erdman = ATCC ...,Mycobacterium tuberculosis ATCC 35801 = str. E...,604194|604195|604196|604197|604198|604201|6042...,H8EVV4|H8F0D6|H8F0D7|H8F1Z2|H8F3Q9|H8F3R6,NaN,NaN,22535945,UniProt:652616,10.1128/jb.00353-12,PMC3347172,18777523,"Miyoshi-Akiyama T, Matsumura K, Iwai H, Funato..."
3,1806,BioAssay|Pathway|Cooccurrence,1,Mycobacterium tuberculosis variant microti,Mycobacterium microti Reed 1957 (Approved List...,392601,NaN,NaN,NaN,29205127,NCIt:C85548|Wikidata:Q4752705,10.1099/ijsem.0.002507,NaN,27028446,"Riojas MA, McGough KJ, Rider-Riojas CJ, Rastog..."
4,115862,Cooccurrence,1,Mycobacterium tuberculosis variant caprae,Mycobacterium bovis subsp. caprae (Aranaz et a...,NaN,NaN,NaN,NaN,10618079|10425790|11931153|14657105|29205127,Wikidata:Q6946997,10.1099/00207713-49-3-1263|10.1099/00207713-52...,PMC86043,4311809|8436056|11807723|26781904|27028446,"Riojas MA, McGough KJ, Rider-Riojas CJ, Rastog..."


## 2. Select Relevant Columns & Remove Empty Entries

In [47]:
df = (
    df_raw[["Taxonomy_ID", "Taxonomy_Name", "Linked_BioAssays"]]
    .dropna(subset=["Linked_BioAssays"])
)

df.head()

,Taxonomy_ID,Taxonomy_Name,Linked_BioAssays
0,1773,Mycobacterium tuberculosis,375|838|1811|38730|38912|38913|40463|40464|103...
1,1765,Mycobacterium tuberculosis variant bovis,143878|143879|143880|143993|143994|143995|1439...
2,652616,Mycobacterium tuberculosis str. Erdman = ATCC ...,604194|604195|604196|604197|604198|604201|6042...
3,1806,Mycobacterium tuberculosis variant microti,392601
5,419947,Mycobacterium tuberculosis H37Ra,440721|584529|661408|1296444|1296445|1318545|1...


## 3. Expand BioAssays into Individual AIDs

In [48]:
df["AID"] = df["Linked_BioAssays"].str.split("|")
df_expanded = df.explode("AID").copy()
df_expanded["AID"] = df_expanded["AID"].astype(int)

df_final = df_expanded[["Taxonomy_ID", "Taxonomy_Name", "AID"]].drop_duplicates()
df_final.head()

,Taxonomy_ID,Taxonomy_Name,AID
0,1773,Mycobacterium tuberculosis,375
0,1773,Mycobacterium tuberculosis,838
0,1773,Mycobacterium tuberculosis,1811
0,1773,Mycobacterium tuberculosis,38730
0,1773,Mycobacterium tuberculosis,38912


## 4. Count Assays per Taxonomy ID

In [51]:
assays_per_tax = (
    df_final.groupby(["Taxonomy_ID", "Taxonomy_Name"])["AID"]
    .nunique()
    .reset_index(name="N_Assays")
    .sort_values("N_Assays", ascending=False)
    .reset_index(drop=True)
)

assays_per_tax

,Taxonomy_ID,Taxonomy_Name,N_Assays
0,1773,Mycobacterium tuberculosis,1431
1,83332,Mycobacterium tuberculosis H37Rv,1246
2,1764,Mycobacterium avium,612
3,1765,Mycobacterium tuberculosis variant bovis,307
4,33892,Mycobacterium tuberculosis variant bovis BCG,246
5,83331,Mycobacterium tuberculosis CDC1551,80
6,410289,Mycobacterium tuberculosis variant bovis BCG s...,28
7,419947,Mycobacterium tuberculosis H37Ra,23
8,998092,Mycobacterium tuberculosis variant bovis BCG s...,15
9,652616,Mycobacterium tuberculosis str. Erdman = ATCC ...,14


## 5. Summary

In [50]:
summary = pd.DataFrame({
    "Metric": ["Unique Taxonomy IDs", "Unique Assay IDs"],
    "Value": [
        df_final["Taxonomy_ID"].nunique(),
        df_final["AID"].nunique()
    ]
})

summary

,Metric,Value
0,Unique Taxonomy IDs,16
1,Unique Assay IDs,3808
